# Rule-based vs ML: comparison on the same test set (v5 dataset)

We compare the **full** rule-based engine (`fraud_checks/services.py`,
all 6 signals) against the ML model on the same test set (v5).
Rules 1–6 cover: account_blocked, insufficient_balance,
limit_exceeded (daily > 200k), high_amount (> 100k),
new_account (< 7d), high_frequency (> 10tx/h).
ML uses threshold 0.59 (tuned on val set, see rf_tuning.ipynb).

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..", ".."))
from ml.synthetic.build_dataset import generate_accounts

DATASET_PATH = "../datasets/synthetic_v5.csv"
MODEL_PATH = "../models/fraud_model_v5.pkl"
FEATURE_COLS = ["amount", "account_age_days", "tx_last_hour", "transaction_hour",
              "receiver_tx_count_24h", "sender_daily_amount_sum",
              "amount_to_balance_ratio", "days_since_last_tx"]
BEST_THRESHOLD = 0.59

df = pd.read_csv(DATASET_PATH, parse_dates=["created_at"])
df = df.sort_values("created_at").reset_index(drop=True)

# Re-generate accounts (deterministic, SEED=42) for balance info
from ml.synthetic import config
accounts_df = generate_accounts(config.N_ACCOUNTS, np.random.default_rng(config.SEED))

# same time-based split as in train_model.py (train_ratio=0.8)
split_idx = int(len(df) * 0.8)
test_df = df.iloc[split_idx:].copy()

model = joblib.load(MODEL_PATH)
print(f"Test set: {len(test_df)} rows ({test_df["is_fraud"].mean():.2%} fraud)")

Test set: 4000 rows (3.02% fraud)


## Step 1 — ML model predictions

Using tuned threshold from rf_tuning.ipynb (0.59).

In [2]:
X_test = test_df[FEATURE_COLS]
test_df["ml_proba"] = model.predict_proba(X_test)[:, 1]
test_df["ml_pred"] = (test_df["ml_proba"] >= BEST_THRESHOLD).astype(int)

## Step 2 — Rule-based engine (full)

Logic ported from `fraud_checks/services.py` `calculate_risk()`.
All 6 signals are computed from DataFrame columns:
- account_blocked: is_blocked (False for all synthetic)
- insufficient_balance: amount > sender_balance
- limit_exceeded: sender_daily_amount_sum + amount > 200k
- high_amount: amount > 100k
- high_frequency: tx_last_hour > 10
- new_account: account_age_days < 7

In [3]:
def rule_based_full(row):
    risk_score = 0
    reasons = []
    if row["account_age_days"] < 7:
        risk_score += 20
        reasons.append("new_account")
    if row["amount"] > 100000:
        risk_score += 40
        reasons.append("high_amount")
    if row["tx_last_hour"] > 10:
        risk_score += 30
        reasons.append("high_frequency")
    if row["sender_daily_amount_sum"] + row["amount"] > 200000:
        risk_score += 60
        reasons.append("limit_exceeded")
    if row["amount"] > row["sender_balance_before"]:
        risk_score += 80
        reasons.append("insufficient_balance")
    # account_blocked: is_blocked is False for all synthetic, never fires

    if risk_score >= 60:
        decision = "BLOCKED"
    elif risk_score >= 30:
        decision = "REVIEW"
    else:
        decision = "APPROVED"
    return risk_score, decision, reasons

# Add sender balance column for insufficient_balance rule
balance_map = accounts_df.set_index("account_id")["balance"]
test_df["sender_balance_before"] = test_df["sender_account_id"].map(balance_map)

results = test_df.apply(rule_based_full, axis=1, result_type="expand")
test_df["rule_risk_score"] = results[0]
test_df["rule_decision"] = results[1]
test_df["rule_reasons"] = results[2]
# REVIEW and BLOCKED count as "system flagged something" -> 1
test_df["rule_pred"] = test_df["rule_decision"].isin(["BLOCKED", "REVIEW"]).astype(int)

## Step 3 — Metric comparison (full rules vs ML, with recall by pattern)

In [4]:
print("=== Rule-based (full) ===\n")
print(classification_report(test_df["is_fraud"], test_df["rule_pred"],
                              target_names=["normal", "fraud"]))

print("\n=== ML model ===\n")
print(classification_report(test_df["is_fraud"], test_df["ml_pred"],
                              target_names=["normal", "fraud"]))

# --- Recall by fraud_pattern ---
print("\n=== Recall by fraud_pattern ===\n")
for system_name, pred_col in [("Rules", "rule_pred"), ("ML", "ml_pred")]:
    print(f"--- {system_name} ---")
    for pattern in sorted(test_df["fraud_pattern"].dropna().unique()):
        mask = test_df["fraud_pattern"] == pattern
        total = int(mask.sum())
        caught = int(((test_df.loc[mask, "is_fraud"] == 1) & (test_df.loc[mask, pred_col] == 1)).sum())
        recall = caught / total if total > 0 else 0
        print(f"  {pattern:35s}: {caught:3d}/{total}  (recall={recall:.2%})")
    print()

# --- Feature importances ---
print("=== Feature importances ===\n")
importances = pd.Series(
    model.feature_importances_, index=FEATURE_COLS
).sort_values(ascending=False)
print(importances)

=== Rule-based (full) ===

              precision    recall  f1-score   support

      normal       0.98      0.81      0.89      3879
       fraud       0.08      0.55      0.15       121

    accuracy                           0.81      4000
   macro avg       0.53      0.68      0.52      4000
weighted avg       0.96      0.81      0.87      4000


=== ML model ===

              precision    recall  f1-score   support

      normal       0.99      0.99      0.99      3879
       fraud       0.77      0.73      0.75       121

    accuracy                           0.98      4000
   macro avg       0.88      0.86      0.87      4000
weighted avg       0.98      0.98      0.98      4000


=== Recall by fraud_pattern ===

--- Rules ---
  balance_drain_fraud                :   0/19  (recall=0.00%)
  dormant_reactivation_fraud         :  11/12  (recall=91.67%)
  mule_fraud                         :   6/27  (recall=22.22%)
  new_account_fraud                  :  17/20  (recall=85.00%)
 

amount                     0.411828
amount_to_balance_ratio    0.221811
receiver_tx_count_24h      0.108290
days_since_last_tx         0.103104
account_age_days           0.063019
transaction_hour           0.043409
sender_daily_amount_sum    0.040523
tx_last_hour               0.008017
dtype: float64


## Step 4 — Where they disagree

The most informative part: examining specific cases of disagreement.

In [5]:
# ML catches, rule misses (ML better here)
ml_catches_rule_misses = test_df[
    (test_df["is_fraud"] == 1) &
    (test_df["rule_pred"] == 0) &
    (test_df["ml_pred"] == 1)
]
print(f"ML caught fraud that rules missed: {len(ml_catches_rule_misses)}")
disp_cols = ["amount", "account_age_days", "tx_last_hour",
             "receiver_tx_count_24h", "sender_daily_amount_sum",
             "amount_to_balance_ratio", "days_since_last_tx",
             "fraud_pattern"]

print(ml_catches_rule_misses[disp_cols].head(10))

# Rule catches, ML misses (rule better here)
rule_catches_ml_misses = test_df[
    (test_df["is_fraud"] == 1) &
    (test_df["rule_pred"] == 1) &
    (test_df["ml_pred"] == 0)
]
print(f"\nRules caught fraud that ML missed: {len(rule_catches_ml_misses)}")
print(rule_catches_ml_misses[disp_cols].head(10))

# Both missed (most dangerous — neither system reacted)
both_miss = test_df[
    (test_df["is_fraud"] == 1) &
    (test_df["rule_pred"] == 0) &
    (test_df["ml_pred"] == 0)
]
print(f"\nBoth missed: {len(both_miss)}")
print(both_miss[disp_cols].head(10))

ML caught fraud that rules missed: 31
         amount  account_age_days  tx_last_hour  receiver_tx_count_24h  \
16160  51963.09               344             2                      0   
16250   7418.44               269             0                      2   
16251   4912.73                17             0                      3   
16257  24718.23               345             1                      0   
16341  11746.34               595             0                      5   
16343   2601.97               102             0                      6   
16400   5520.07               360             0                      2   
16403  14655.32               644             0                      4   
16404    688.86               466             0                      5   
16414  52423.34               181             0                      0   

       sender_daily_amount_sum  amount_to_balance_ratio  days_since_last_tx  \
16160                 77085.51                   0.9102            0

## Conclusions (v5 data, threshold=0.59, 121 fraud in test)

### Comparison summary (test set, 4000 rows)

| Category | Count |
|---|---|
| Both caught (ML+Rules agree on fraud) | 57 |
| ML better (rules miss, ML catches) | 31 |
| Rules better (ML miss, rules catch) | 9 |
| Both missed | 24 |

### Recall by fraud pattern

| Pattern | Rules recall | ML recall @ 0.59 | n_test |
|---|---|---|---|
| new_account_fraud | 85% | 75% | 20 |
| structuring_fraud | 78% | 100% | 23 |
| velocity_fraud | 70% | 85% | 20 |
| dormant_reactivation_fraud | 92% | 58% | 12 |
| mule_fraud | 22% | 74% | 27 |
| balance_drain_fraud | 0% | 32% | 19 |

### Key takeaways

- ML dominates mule_fraud (74% vs 22%), structuring (100% vs 78%), velocity (85% vs 70%).
- Rules still better on dormant_reactivation (92% vs 58%) and new_account (85% vs 75%).
- Overall: ML catches 31 frauds that rules miss; rules catch 9 that ML misses; 24 missed by both.
- Feature ranking: amount (0.41) > amt_to_balance_ratio (0.22) > receiver_tx_count_24h (0.11).

### Known limitations

**balance_drain_fraud**: blind spot for both systems — rules (0%) and ML (32% on sklearn 1.9.0).
Recall varies 11–32% between sklearn 1.8.0 and 1.9.0 with identical code/data/random_state.
Root cause: amount_to_balance_ratio overlaps with normal transactions (see eda_feature_distributions.ipynb).

**dormant_reactivation_fraud**: ML (58%) significantly behind rules (92%) — rules catch it accidentally
via the `new_account` heuristic (<7 days) which overlaps with dormant reactivation profiles.

Both require additional features or a different model family — out of scope for this iteration.

### Reproducibility

Results depend on sklearn version. Pinned to `scikit-learn==1.9.0` in requirements.txt.
